**Capítulo 7 - Aprendizado Ensemble e Florestas Aleatórias**

*Este notebook mostra como combinar vários modelos para obter previsões melhores: votação, bagging, pasting, Florestas Aleatórias, importância de atributos, boosting, gradient boosting e stacking.*

*Este notebook contém a tradução e adaptação das células de exemplo do Capítulo 7. A seção final de soluções dos exercícios foi deixada fora do notebook principal e ficará organizada em `Respostas.md`.*

<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/ageron/handson-ml3/blob/main/07_ensemble_learning_and_random_forests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>
  </td>
  <td>
    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/ageron/handson-ml3/blob/main/07_ensemble_learning_and_random_forests.ipynb"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" /></a>
  </td>
</table>

# Configuração

In [13]:
import sys
import sklearn
from packaging import version

assert sys.version_info >= (3, 7)
assert version.parse(sklearn.__version__) >= version.parse("1.0.1")

In [14]:
from sklearn.datasets import fetch_openml

mnist = fetch_openml('mnist_784',as_frame=False)

In [15]:
from sklearn.model_selection import train_test_split

X = mnist.data / 255.0
y = mnist.target.astype("uint8")

X_train_valid, X_test, y_train_valid, y_test = train_test_split(
    X,
    y,
    test_size=10_000,
    random_state=42,
    stratify=y
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_valid,
    y_train_valid,
    test_size=10_000,
    random_state=42,
    stratify=y_train_valid
)

X_train.shape, X_valid.shape, X_test.shape

((50000, 784), (10000, 784), (10000, 784))

In [16]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

random_forest_clf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

extra_trees_clf = ExtraTreesClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

log_reg_clf = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1_000, random_state=42, n_jobs=-1)
)

In [17]:
estimators = [
    ("Random Forest", random_forest_clf),
    ("Extra Trees", extra_trees_clf),
    ("Logistic Regression", log_reg_clf)
]

for name, estimator in estimators:
    print("Treinando:", name)
    estimator.fit(X_train, y_train)

Treinando: Random Forest
Treinando: Extra Trees
Treinando: Logistic Regression


In [18]:
from sklearn.metrics import accuracy_score

valid_scores = []

for name, estimator in estimators:
    y_pred = estimator.predict(X_valid)
    acc = accuracy_score(y_valid, y_pred)
    valid_scores.append((name, acc))
    print(f"{name}: {acc:.4f}")

Random Forest: 0.9710
Extra Trees: 0.9731
Logistic Regression: 0.9175


In [19]:
best_valid_name, best_valid_score = max(valid_scores, key=lambda score: score[1])

print("Melhor classificador individual na validacao:", best_valid_name)
print(f"Acuracia de validacao: {best_valid_score:.4f}")

Melhor classificador individual na validacao: Extra Trees
Acuracia de validacao: 0.9731


In [20]:
from sklearn.ensemble import VotingClassifier

voting_clf = VotingClassifier(
    estimators=[
        ("random_forest", random_forest_clf),
        ("extra_trees", extra_trees_clf),
        ("log_reg", log_reg_clf)
    ],
    voting="soft"
)

voting_clf.fit(X_train, y_train)

y_valid_pred = voting_clf.predict(X_valid)
voting_valid_acc = accuracy_score(y_valid, y_valid_pred)

print(f"Voting Classifier na validacao: {voting_valid_acc:.4f}")

Voting Classifier na validacao: 0.9539


In [21]:
y_test_pred = voting_clf.predict(X_test)
voting_test_acc = accuracy_score(y_test, y_test_pred)

print(f"Voting Classifier no teste: {voting_test_acc:.4f}")

Voting Classifier no teste: 0.9501


In [22]:
test_scores = []

for name, estimator in estimators + [("Voting Classifier", voting_clf)]:
    y_test_pred = estimator.predict(X_test)
    acc = accuracy_score(y_test, y_test_pred)
    test_scores.append((name, acc))
    print(f"{name}: {acc:.4f}")

Random Forest: 0.9657
Extra Trees: 0.9706
Logistic Regression: 0.9149
Voting Classifier: 0.9501


In [23]:
individual_test_scores = test_scores[:-1]
best_name, best_score = max(individual_test_scores, key=lambda score: score[1])

improvement = voting_test_acc - best_score

print("Melhor individual no teste:", best_name)
print(f"Acuracia do melhor individual: {best_score:.4f}")
print(f"Acuracia do Voting Classifier: {voting_test_acc:.4f}")
print(f"Melhoria: {improvement:.4f}")
print(f"Melhoria em pontos percentuais: {improvement * 100:.2f} p.p.")

Melhor individual no teste: Extra Trees
Acuracia do melhor individual: 0.9706
Acuracia do Voting Classifier: 0.9501
Melhoria: -0.0205
Melhoria em pontos percentuais: -2.05 p.p.
